In [2]:
# robust_cars24_scraper_edge.py
import time
import random
import pandas as pd
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ---------------- CONFIG ----------------
EDGE_DRIVER_PATH = r"C:/Drivers/msedgedriver.exe"   # <-- update this to your msedgedriver path
LISTING_URL = "https://www.cars24.com/buy-used-maruti-cars-mumbai/"  # listing page
MAX_SCROLL_ATTEMPTS_WITHOUT_NEW = 6  # stop after this many scroll tries that produce no new links
MAX_TOTAL_SCROLLS = 120               # safety upper bound (avoid infinite loop)
SCROLL_PAUSE_MIN = 1.0
SCROLL_PAUSE_MAX = 2.5
# ----------------------------------------

# Setup Edge
options = webdriver.EdgeOptions()
options.add_argument("--start-maximized")
# options.add_argument("--disable-blink-features=AutomationControlled")  # optional
driver = webdriver.Edge(service=Service(EDGE_DRIVER_PATH), options=options)
wait = WebDriverWait(driver, 20)

def close_cookie_or_popup():
    # Try to close cookie/banner/popups that could block scrolling or links
    try:
        # look for common button text
        btns = driver.find_elements(By.TAG_NAME, "button")
        for b in btns:
            try:
                t = (b.text or "").strip().lower()
                if any(x in t for x in ("accept", "agree", "allow", "ok", "got it", "close", "dismiss")):
                    b.click()
                    time.sleep(0.5)
                    return
            except:
                continue
    except:
        pass

def collect_links_from_wrapper():
    links = set()
    try:
        anchors = driver.find_elements(By.CSS_SELECTOR, "div.styles_carListingContainer__uob_6 div.styles_wrapper__b4UUV a")
        for a in anchors:
            href = a.get_attribute("href")
            if href and "/buy-used-" in href:
                links.add(href.split("?")[0])   # strip query params if any
    except Exception as e:
        # fallback: find any a with buy-used
        anchors = driver.find_elements(By.CSS_SELECTOR, "a[href*='/buy-used-']")
        for a in anchors:
            try:
                href = a.get_attribute("href")
                if href and "cars24.com" in href and "/buy-used-" in href:
                    links.add(href.split("?")[0])
            except:
                continue
    return links

def try_click_load_more():
    # search for common "Load more" / "Show more" buttons and click
    try:
        possible = driver.find_elements(By.XPATH,
            "//button[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'load more')"
            " or contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'show more')"
            " or contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'view more')"
            " or contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'see more')]")
        for btn in possible:
            try:
                if btn.is_displayed():
                    btn.click()
                    time.sleep(1.0)
                    return True
            except:
                continue
    except:
        pass
    return False

# Open listing page
driver.get(LISTING_URL)
time.sleep(5)                 # initial wait for JS/bootstrap
close_cookie_or_popup()

# wait for wrapper or at least an <a> to appear
try:
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.styles_carListingContainer__uob_6")))
except:
    # still continue; sometimes waiting fails but page still has cards
    pass

# iterative scrolling loop: scroll to last anchor, wait, collect links, stop when stable
collected_links = set()
attempts_no_change = 0
total_scrolls = 0
last_count = 0

while attempts_no_change < MAX_SCROLL_ATTEMPTS_WITHOUT_NEW and total_scrolls < MAX_TOTAL_SCROLLS:
    total_scrolls += 1

    # get current anchors and scroll last into view if present
    anchors = driver.find_elements(By.CSS_SELECTOR, "div.styles_carListingContainer__uob_6 div.styles_wrapper__b4UUV a")
    if anchors:
        try:
            last_anchor = anchors[-1]
            # scroll that anchor into view (better than scrolling page bottom)
            driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'end'});", last_anchor)
        except:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    else:
        # fallback: page bottom
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

    # small random pause to mimic human browsing and allow XHR to load
    time.sleep(random.uniform(SCROLL_PAUSE_MIN, SCROLL_PAUSE_MAX))

    # try clicking any 'load more' if present
    try_click_load_more()

    # collect links
    new_links = collect_links_from_wrapper()
    current_count = len(new_links)
    print(f"[scroll {total_scrolls}] links found: {current_count}")

    if current_count > last_count:
        # new links arrived
        collected_links = new_links
        last_count = current_count
        attempts_no_change = 0
    else:
        attempts_no_change += 1

    # small additional wait every few iterations
    if total_scrolls % 10 == 0:
        time.sleep(2.0)

print(f"\nFinished scrolling. Total unique detail links found: {len(collected_links)}")

# If still small (≈20), print a hint for debugging
if len(collected_links) <= 20:
    print("⚠️ Only ~20 links found. Possible reasons:\n"
          "- Cars24 may serve a small initial batch and require user interaction or filters to load more\n"
          "- Cloudflare/bot-detection may be preventing further XHRs\n"
          "- The listing page might limit results without further navigation\n"
          "Try opening the listing page in a normal browser and scroll manually to confirm more items load.\n")

# ---------------- SCRAPE DETAIL PAGES ----------------
data_rows = []
for i, url in enumerate(sorted(collected_links), start=1):
    try:
        driver.get(url)
        time.sleep(3.0)   # wait for page JS to render

        # Basic fields
        try:
            title_el = driver.find_element(By.CSS_SELECTOR, "h1.sc-braxZu.fjhfdl")
            title_text = title_el.text.strip()
        except:
            title_text = ""

        year = ""
        if title_text:
            # parse year and model (assumes "2016 Maruti Alto K10")
            parts = title_text.split()
            if parts and parts[0].isdigit():
                year = parts[0]

        # meta p tags (ordered)
        kilometer_driven = number_of_owners = transmission = fuel_type = reg_no = ""
        try:
            meta_ps = driver.find_elements(By.CSS_SELECTOR, "div.styles_carMeta__hm1XQ p.sc-braxZu.kvfdZL")
            if len(meta_ps) > 0: kilometer_driven = meta_ps[0].text.strip()
            if len(meta_ps) > 1: number_of_owners = meta_ps[1].text.strip()
            if len(meta_ps) > 2: transmission = meta_ps[2].text.strip()
            if len(meta_ps) > 3: fuel_type = meta_ps[3].text.strip()
        except:
            pass

        # location
        try:
            location = driver.find_element(By.CSS_SELECTOR, "div.styles_carLocation__UrZVn p.sc-braxZu.dughoY").text.strip()
        except:
            location = ""

        # price
        price = ""
        try:
            price = driver.find_element(By.CSS_SELECTOR, "div.styles_price__3yE9i p.sc-braxZu.hhzsvw").text.strip()
        except:
            # fallback: any price-like p inside price wrapper
            try:
                price = driver.find_element(By.CSS_SELECTOR, "div.styles_price__3yE9i p").text.strip()
            except:
                price = ""

        data_rows.append({
            "Year": year,
            "Kilometer Driven": kilometer_driven,
            "Number of Owners": number_of_owners,
            "Transmission": transmission,
            "Fuel Type": fuel_type,
            "Location": location,
            "Price": price
        })
    except Exception as e:
        print(f"  Error scraping {url}: {e}")

# Save results
df = pd.DataFrame(data_rows)
df.to_excel("maruti_mumbai_cars_full(1).xlsx", index=False)
print(f"\nSaved {len(df)} rows to maruti_mumbai_cars_full.xlsx")

driver.quit()


[scroll 1] links found: 40
[scroll 2] links found: 60
[scroll 3] links found: 80
[scroll 4] links found: 100
[scroll 5] links found: 120
[scroll 6] links found: 140
[scroll 7] links found: 160
[scroll 8] links found: 180
[scroll 9] links found: 200
[scroll 10] links found: 220
[scroll 11] links found: 240
[scroll 12] links found: 260
[scroll 13] links found: 280
[scroll 14] links found: 300
[scroll 15] links found: 320
[scroll 16] links found: 340
[scroll 17] links found: 360
[scroll 18] links found: 380
[scroll 19] links found: 393
[scroll 20] links found: 393
[scroll 21] links found: 393
[scroll 22] links found: 393
[scroll 23] links found: 393
[scroll 24] links found: 393
[scroll 25] links found: 393

Finished scrolling. Total unique detail links found: 393
  Error scraping https://www.cars24.com/buy-used-maruti-new-wagon-r-2020-cars-mumbai-10396205710/: HTTPConnectionPool(host='localhost', port=51107): Read timed out. (read timeout=120)
  Error scraping https://www.cars24.com/buy-u